# CreditGuard — Credit Risk & Default Prediction

**Dataset:** Give Me Some Credit (`cs-training.csv`)  
**Goal:** Predict whether a borrower will experience serious delinquency within two years and turn the model output into an interpretable credit-risk score.

### End-to-end architecture

```text
RAW DATA
   ↓
Data Validation
   ↓
EDA
   ↓
Missing Values + Outliers
   ↓
Feature Engineering
   ↓
Stratified Train / Validation / Test Split
   ↓
ML Pipeline
   ├── Dummy Classifier
   ├── Logistic Regression
   └── Linear SVM
   ↓
Model Comparison
   ↓
ROC-AUC / PR-AUC / Precision / Recall / F1
   ↓
Threshold Tuning
   ↓
Default Probability
   ↓
Risk Segmentation
   ↓
Business Interpretation
```

### Why a validation set?
The original architecture says train/test, but threshold tuning must **not** use the final test set. Therefore this notebook uses a stratified **70% train / 15% validation / 15% test** split. The validation set chooses the threshold; the test set is touched only for final evaluation.

> The notebook intentionally stays interview-friendly: no XGBoost or Random Forest. The focus is on data quality, class imbalance, Logistic Regression, Linear SVM, evaluation, threshold selection and business interpretation.

## 1. Imports and reproducibility

Google Colab already includes the packages used below, so no installation step is normally required.

In [ ]:
from pathlib import Path
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 2. Load the dataset

### Colab instructions
Upload `cs-training.csv` into the Colab session when prompted. If the file is already present in `/content`, the notebook finds it automatically.

In [ ]:
def find_training_file():
    candidates = [
        Path('/content/cs-training.csv'),
        Path('/mnt/data/cs-training.csv'),
        Path('cs-training.csv'),
    ]
    for path in candidates:
        if path.exists():
            return path

    try:
        from google.colab import files
        print('cs-training.csv was not found. Please upload it now.')
        uploaded = files.upload()
        if 'cs-training.csv' not in uploaded:
            raise FileNotFoundError('Please upload a file named cs-training.csv')
        return Path('/content/cs-training.csv')
    except ImportError:
        raise FileNotFoundError(
            'cs-training.csv not found. Put it in the current folder and rerun this cell.'
        )

DATA_PATH = find_training_file()
df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded: {DATA_PATH}')
print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head()

## 3. Data validation

We first verify the schema, target, missing values and suspicious values **before modelling**.

The first CSV column is only a row identifier, so it is not a predictive feature.

In [ ]:
print('Columns:')
for col in df_raw.columns:
    print(' -', col)

TARGET = 'SeriousDlqin2yrs'
ID_COL = 'Unnamed: 0'

assert TARGET in df_raw.columns, f'Missing target column: {TARGET}'
assert set(df_raw[TARGET].dropna().unique()).issubset({0, 1}), 'Target is not binary.'
assert df_raw[TARGET].isna().sum() == 0, 'Target contains missing values.'

print('\nTarget counts:')
print(df_raw[TARGET].value_counts().sort_index())
print('\nTarget percentages:')
print((df_raw[TARGET].value_counts(normalize=True).sort_index() * 100).round(2))

print('\nMissing values:')
missing = df_raw.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

print('\nExact duplicates including row ID:', df_raw.duplicated().sum())
print('Duplicate feature/target profiles after excluding row ID:',
      df_raw.drop(columns=[ID_COL]).duplicated().sum())

### Data-quality checks

Two useful observations from this dataset:

- `age` contains an impossible value of `0`.
- The three delinquency-count columns contain unusually large repeated values `96` and `98`. Because these are extreme and appear consistently across all three delinquency fields, this project treats them as **anomalous coded values** rather than genuine counts. This is a modelling choice based on inspection, not a claim from the data dictionary.

We keep duplicate feature profiles because different borrowers can legitimately have identical observed features.

In [ ]:
delinquency_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse',
]

print('age <= 0:', int((df_raw['age'] <= 0).sum()))
for col in delinquency_cols:
    print(f'{col}: values 96/98 = {int(df_raw[col].isin([96, 98]).sum())}')

## 4. Exploratory Data Analysis (EDA)

### 4.1 Target imbalance
A majority-class accuracy can look high even if the model detects **zero defaulters**, so accuracy alone is not a useful primary metric here.

In [ ]:
target_counts = df_raw[TARGET].value_counts().sort_index()
plt.figure(figsize=(6, 4))
target_counts.plot(kind='bar')
plt.title('Target Distribution')
plt.xlabel('SeriousDlqin2yrs (0 = No, 1 = Yes)')
plt.ylabel('Borrowers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

positive_rate = df_raw[TARGET].mean()
print(f'Positive/default-class rate: {positive_rate:.2%}')

### 4.2 Missing-value profile

In [ ]:
missing_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(8, 4))
missing_pct.plot(kind='bar')
plt.title('Missing Values by Feature')
plt.ylabel('Missing (%)')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

missing_pct.to_frame('missing_percent')

### 4.3 Summary statistics and outlier inspection

The table below makes the extreme tails visible. We will not blindly delete every large value; instead, we will make a small number of transparent cleaning decisions and fit any clipping limits using **training data only**.

In [ ]:
feature_cols_raw = [c for c in df_raw.columns if c not in [ID_COL, TARGET]]
summary = df_raw[feature_cols_raw].describe(
    percentiles=[0.01, 0.50, 0.95, 0.99, 0.995]
).T
summary[['count', 'mean', 'std', 'min', '1%', '50%', '95%', '99%', '99.5%', 'max']]

### 4.4 Selected feature distributions

In [ ]:
def plot_histogram(data, column, bins=50, upper_quantile=None):
    values = data[column].dropna()
    if upper_quantile is not None:
        upper = values.quantile(upper_quantile)
        values = values[values <= upper]
    plt.figure(figsize=(7, 4))
    plt.hist(values, bins=bins)
    plt.title(f'Distribution: {column}')
    plt.xlabel(column)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

plot_histogram(df_raw, 'age', bins=40)
plot_histogram(df_raw, 'MonthlyIncome', bins=50, upper_quantile=0.99)
plot_histogram(df_raw, 'DebtRatio', bins=50, upper_quantile=0.99)
plot_histogram(df_raw, 'RevolvingUtilizationOfUnsecuredLines', bins=50, upper_quantile=0.99)

### 4.5 Correlation overview

Correlation is only a **linear association measure**. A low correlation does not mean a feature is useless.

In [ ]:
corr_df = df_raw.drop(columns=[ID_COL]).corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr_df, aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr_df.columns)), corr_df.columns, rotation=90)
plt.yticks(range(len(corr_df.index)), corr_df.index)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Cleaning and feature engineering

Cleaning choices:

1. Remove the row-ID column.
2. Convert `age <= 0` to missing so it can be imputed from training data.
3. Convert delinquency values `96` and `98` to missing because they behave like anomalous codes in these count fields.
4. Add only a few interpretable engineered features.

### Engineered features

- `TotalDelinquencies` = sum of 30–59, 60–89 and 90+ day delinquency counts.
- `IncomePerDependent` = monthly income divided by dependents + 1.
- `HasRealEstateLoan` = indicator for at least one real-estate loan/line.

In [ ]:
def basic_clean_and_engineer(data):
    data = data.copy()

    if ID_COL in data.columns:
        data = data.drop(columns=[ID_COL])

    # Impossible age -> missing
    data.loc[data['age'] <= 0, 'age'] = np.nan

    # Observed anomalous codes in all three delinquency count fields
    for col in delinquency_cols:
        data[col] = data[col].replace({96: np.nan, 98: np.nan})

    # Simple, interpretable engineered features
    data['TotalDelinquencies'] = data[delinquency_cols].sum(axis=1, min_count=1)
    data['IncomePerDependent'] = (
        data['MonthlyIncome'] / (data['NumberOfDependents'] + 1)
    )
    data['HasRealEstateLoan'] = (
        data['NumberRealEstateLoansOrLines'] > 0
    ).astype(int)

    return data

model_df = basic_clean_and_engineer(df_raw)
print('Model dataframe shape:', model_df.shape)
model_df.head()

## 6. Stratified train / validation / test split

We split **before learning imputation values, clipping limits or scaling parameters**. This is one of the most important leakage-prevention steps in the project.

In [ ]:
X = model_df.drop(columns=[TARGET])
y = model_df[TARGET].astype(int)

# 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Split temporary set equally -> 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    'rows': [len(X_train), len(X_val), len(X_test)],
    'positive_rate': [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=['train', 'validation', 'test'])

split_summary

## 7. Outlier handling — fitted on training data only

For a few highly skewed continuous variables, we cap only the extreme 0.5% tails using limits learned from the training set. The same limits are then applied to validation and test data.

This prevents the test set from influencing preprocessing.

In [ ]:
clip_cols = [
    'RevolvingUtilizationOfUnsecuredLines',
    'DebtRatio',
    'MonthlyIncome',
    'IncomePerDependent',
]

lower_bounds = X_train[clip_cols].quantile(0.005)
upper_bounds = X_train[clip_cols].quantile(0.995)


def apply_training_clips(data):
    data = data.copy()
    data.loc[:, clip_cols] = data[clip_cols].clip(
        lower=lower_bounds,
        upper=upper_bounds,
        axis=1,
    )
    return data

X_train = apply_training_clips(X_train)
X_val = apply_training_clips(X_val)
X_test = apply_training_clips(X_test)

pd.DataFrame({'lower_bound': lower_bounds, 'upper_bound': upper_bounds})

## 8. Build the ML pipelines

### Why pipelines?
Each model gets preprocessing that is fitted only on the training fold:

```text
Median Imputation → Standardisation → Model
```

`StandardScaler` is especially important for Logistic Regression and Linear SVM because their optimisation depends on feature scale.

### Models

- **Dummy Classifier:** majority-class baseline.
- **Logistic Regression:** main interpretable probability model.
- **Linear SVM:** maximum-margin comparison model; we use the linear version because the dataset is large and low-dimensional.

In [ ]:
def make_scaled_pipeline(model):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model),
    ])

models = {
    'Dummy Classifier': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', DummyClassifier(strategy='most_frequent')),
    ]),
    'Logistic Regression': make_scaled_pipeline(
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    ),
    'Linear SVM': make_scaled_pipeline(
        LinearSVC(C=1.0, dual=False, max_iter=5000, random_state=RANDOM_STATE)
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'Fitted: {name}')

## 9. Model comparison

### Why these metrics?

- **ROC-AUC:** ranking quality across thresholds.
- **PR-AUC / Average Precision:** especially informative for rare positive classes.
- **Precision:** of borrowers flagged risky, how many actually defaulted?
- **Recall:** of actual defaulters, how many did we identify?
- **F1:** balances precision and recall.
- **Balanced Accuracy:** gives equal importance to both classes.

The SVM uses its `decision_function` for ROC-AUC and PR-AUC because a standard Linear SVM does not naturally output probabilities.

In [ ]:
def get_score(model, X_data):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_data)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X_data)
    return model.predict(X_data)


def evaluate_model(model, X_data, y_true):
    pred = model.predict(X_data)
    score = get_score(model, X_data)

    return {
        'ROC_AUC': roc_auc_score(y_true, score),
        'PR_AUC': average_precision_score(y_true, score),
        'Precision': precision_score(y_true, pred, zero_division=0),
        'Recall': recall_score(y_true, pred, zero_division=0),
        'F1': f1_score(y_true, pred, zero_division=0),
        'Balanced_Accuracy': balanced_accuracy_score(y_true, pred),
    }

results = []
for name, model in models.items():
    metrics = evaluate_model(model, X_test, y_test)
    metrics['Model'] = name
    results.append(metrics)

model_metrics = pd.DataFrame(results).set_index('Model').sort_values(
    'ROC_AUC', ascending=False
)
model_metrics

### ROC curves

In [ ]:
plt.figure(figsize=(7, 5))
for name, model in models.items():
    score = get_score(model, X_test)
    fpr, tpr, _ = roc_curve(y_test, score)
    auc = roc_auc_score(y_test, score)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.show()

### Precision–Recall curves

In [ ]:
plt.figure(figsize=(7, 5))
for name, model in models.items():
    score = get_score(model, X_test)
    precision, recall, _ = precision_recall_curve(y_test, score)
    ap = average_precision_score(y_test, score)
    plt.plot(recall, precision, label=f'{name} (AP={ap:.3f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curves')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Logistic Regression at the default 0.50 threshold

Logistic Regression is our main **risk-scoring model** because it naturally outputs probabilities and is easy to interpret.

In [ ]:
logistic_model = models['Logistic Regression']

logistic_test_prob = logistic_model.predict_proba(X_test)[:, 1]
logistic_test_pred_05 = (logistic_test_prob >= 0.50).astype(int)

print('Classification report at threshold = 0.50')
print(classification_report(y_test, logistic_test_pred_05, digits=4, zero_division=0))

cm_05 = confusion_matrix(y_test, logistic_test_pred_05)
cm_05_df = pd.DataFrame(
    cm_05,
    index=['Actual 0', 'Actual 1'],
    columns=['Predicted 0', 'Predicted 1'],
)
cm_05_df

## 11. Threshold tuning — validation set only

A classification threshold of `0.50` is a convention, not a law.

For credit risk:

- **False negative:** an actual defaulter is classified as safe.
- **False positive:** a good borrower is classified as risky.

We choose the threshold that maximises **F1 on the validation set**, then evaluate that frozen threshold once on the test set.

In [ ]:
val_prob = logistic_model.predict_proba(X_val)[:, 1]

precision_vals, recall_vals, thresholds = precision_recall_curve(y_val, val_prob)
precision_for_thresholds = precision_vals[:-1]
recall_for_thresholds = recall_vals[:-1]

f1_vals = (
    2 * precision_for_thresholds * recall_for_thresholds /
    (precision_for_thresholds + recall_for_thresholds + 1e-12)
)

best_idx = int(np.argmax(f1_vals))
best_threshold = float(thresholds[best_idx])

threshold_summary = pd.DataFrame({
    'threshold': thresholds,
    'precision': precision_for_thresholds,
    'recall': recall_for_thresholds,
    'f1': f1_vals,
})

print(f'Best validation threshold: {best_threshold:.4f}')
print(f'Validation precision: {precision_for_thresholds[best_idx]:.4f}')
print(f'Validation recall:    {recall_for_thresholds[best_idx]:.4f}')
print(f'Validation F1:        {f1_vals[best_idx]:.4f}')

### Threshold trade-off

In [ ]:
plot_thresholds = threshold_summary[
    (threshold_summary['threshold'] >= 0.01) &
    (threshold_summary['threshold'] <= 0.60)
]

plt.figure(figsize=(8, 5))
plt.plot(plot_thresholds['threshold'], plot_thresholds['precision'], label='Precision')
plt.plot(plot_thresholds['threshold'], plot_thresholds['recall'], label='Recall')
plt.plot(plot_thresholds['threshold'], plot_thresholds['f1'], label='F1')
plt.axvline(best_threshold, linestyle='--', label=f'Chosen = {best_threshold:.3f}')
plt.xlabel('Classification Threshold')
plt.ylabel('Metric')
plt.title('Threshold Trade-off on Validation Set')
plt.legend()
plt.tight_layout()
plt.show()

## 12. Final test evaluation with the tuned threshold

The test set has not been used to choose the threshold.

In [ ]:
tuned_test_pred = (logistic_test_prob >= best_threshold).astype(int)

comparison = pd.DataFrame({
    'Threshold 0.50': {
        'Precision': precision_score(y_test, logistic_test_pred_05, zero_division=0),
        'Recall': recall_score(y_test, logistic_test_pred_05, zero_division=0),
        'F1': f1_score(y_test, logistic_test_pred_05, zero_division=0),
        'Balanced Accuracy': balanced_accuracy_score(y_test, logistic_test_pred_05),
    },
    f'Tuned {best_threshold:.3f}': {
        'Precision': precision_score(y_test, tuned_test_pred, zero_division=0),
        'Recall': recall_score(y_test, tuned_test_pred, zero_division=0),
        'F1': f1_score(y_test, tuned_test_pred, zero_division=0),
        'Balanced Accuracy': balanced_accuracy_score(y_test, tuned_test_pred),
    },
}).T

comparison

In [ ]:
print(f'ROC-AUC: {roc_auc_score(y_test, logistic_test_prob):.4f}')
print(f'PR-AUC:  {average_precision_score(y_test, logistic_test_prob):.4f}')
print('\nClassification report at tuned threshold')
print(classification_report(y_test, tuned_test_pred, digits=4, zero_division=0))

cm_tuned = confusion_matrix(y_test, tuned_test_pred)
cm_tuned_df = pd.DataFrame(
    cm_tuned,
    index=['Actual 0', 'Actual 1'],
    columns=['Predicted 0', 'Predicted 1'],
)
cm_tuned_df

## 13. Default probability and risk segmentation

The Logistic Regression output is interpreted as an estimated default probability.

For presentation, we create simple **illustrative** risk bands:

- Low: `< 5%`
- Moderate: `5–10%`
- High: `10–20%`
- Very High: `>= 20%`

In a real lender, these cut-offs would be selected using business costs, approval policy, regulation and calibration analysis rather than chosen purely for presentation.

In [ ]:
risk_scores = X_test.copy()
risk_scores['actual_default'] = y_test.values
risk_scores['predicted_default_probability'] = logistic_test_prob
risk_scores['predicted_default_tuned'] = tuned_test_pred

risk_bins = [-np.inf, 0.05, 0.10, 0.20, np.inf]
risk_labels = [
    'Low (<5%)',
    'Moderate (5-10%)',
    'High (10-20%)',
    'Very High (>=20%)',
]

risk_scores['risk_band'] = pd.cut(
    risk_scores['predicted_default_probability'],
    bins=risk_bins,
    labels=risk_labels,
    right=False,
)

risk_summary = (
    risk_scores.groupby('risk_band', observed=True)
    .agg(
        borrowers=('actual_default', 'size'),
        avg_predicted_risk=('predicted_default_probability', 'mean'),
        observed_default_rate=('actual_default', 'mean'),
    )
)

risk_summary

In [ ]:
plt.figure(figsize=(7, 4))
(risk_summary['observed_default_rate'] * 100).plot(kind='bar')
plt.title('Observed Default Rate by Predicted Risk Band')
plt.xlabel('Risk Band')
plt.ylabel('Observed Default Rate (%)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 14. Logistic Regression feature interpretation

Because features are standardised before modelling, each coefficient corresponds to approximately a **one-standard-deviation increase** in that feature, holding the other variables constant.

- Positive coefficient → higher predicted default risk.
- Negative coefficient → lower predicted default risk.
- `exp(coefficient)` gives the corresponding odds multiplier for a one-standard-deviation increase.

This is association within the fitted model, **not causal proof**.

In [ ]:
feature_names = X_train.columns
logistic_estimator = logistic_model.named_steps['model']
coefficients = logistic_estimator.coef_[0]

feature_effects = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'odds_multiplier': np.exp(coefficients),
    'abs_coefficient': np.abs(coefficients),
}).sort_values('abs_coefficient', ascending=False)

feature_effects[['feature', 'coefficient', 'odds_multiplier']].head(15)

In [ ]:
top_effects = feature_effects.head(12).sort_values('coefficient')

plt.figure(figsize=(8, 6))
plt.barh(top_effects['feature'], top_effects['coefficient'])
plt.axvline(0, linewidth=1)
plt.title('Largest Logistic Regression Coefficients')
plt.xlabel('Coefficient (standardised features)')
plt.tight_layout()
plt.show()

## 15. Business interpretation

Use the outputs above to answer four practical questions:

1. **Can the model rank risky borrowers?**  
   Look at ROC-AUC and PR-AUC.

2. **How much risk do we miss at a 0.50 threshold?**  
   Compare recall at 0.50 with the tuned threshold.

3. **What is the trade-off when we lower the threshold?**  
   Recall usually rises, while precision falls.

4. **Do risk bands behave sensibly?**  
   The observed default rate should generally increase from Low to Very High risk.

### Interview-ready project summary

> I built a leakage-safe credit-risk pipeline on the Give Me Some Credit dataset. I validated data quality, handled missing values and anomalous outliers, engineered a few interpretable behavioural features and used stratified train/validation/test splits. I compared a Dummy baseline, Logistic Regression and Linear SVM using ROC-AUC, PR-AUC, precision, recall and F1 rather than relying on accuracy because the default class is rare. Logistic Regression was used as the final risk-scoring model because it provides interpretable probabilities. I tuned the classification threshold on the validation set and converted predicted default probabilities into business-friendly risk bands.

## 16. Save project outputs

The following files are exported:

- `model_metrics.csv`
- `threshold_metrics.csv`
- `threshold_comparison.csv`
- `risk_scores.csv`
- `risk_band_summary.csv`
- `logistic_feature_effects.csv`

They are also packaged into `creditguard_outputs.zip`.

In [ ]:
OUTPUT_DIR = Path('/content/creditguard_outputs') if Path('/content').exists() else Path('creditguard_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_metrics.to_csv(OUTPUT_DIR / 'model_metrics.csv')
threshold_summary.to_csv(OUTPUT_DIR / 'threshold_metrics.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'threshold_comparison.csv')
risk_scores.to_csv(OUTPUT_DIR / 'risk_scores.csv', index=False)
risk_summary.to_csv(OUTPUT_DIR / 'risk_band_summary.csv')
feature_effects.to_csv(OUTPUT_DIR / 'logistic_feature_effects.csv', index=False)

zip_path = OUTPUT_DIR.parent / 'creditguard_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_DIR.iterdir():
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.name)

print(f'Outputs saved to: {OUTPUT_DIR}')
print(f'ZIP created at: {zip_path}')

### Optional: download the output ZIP in Google Colab
Run the next cell only when you want to download the generated outputs.

In [ ]:
# Uncomment these lines in Google Colab when you want to download the ZIP.
# from google.colab import files
# files.download(str(zip_path))

## 17. Key interview questions to be ready for

**Why not accuracy?**  
Because the target is imbalanced. A majority-class model can achieve high accuracy while detecting no defaulters.

**Why stratification?**  
To preserve the class ratio in train, validation and test sets.

**Why median imputation?**  
It is simple and more robust to skewed numerical distributions than mean imputation.

**Why StandardScaler?**  
Logistic Regression and SVM are scale-sensitive; standardisation prevents large-scale variables from dominating optimisation.

**Why Logistic Regression?**  
It is fast, interpretable and naturally provides probabilities, which are useful for credit-risk scoring.

**Why Linear SVM?**  
It provides a maximum-margin comparison model without the computational cost of an RBF-kernel SVM on a large dataset.

**Why tune the threshold?**  
The default 0.50 threshold may not reflect the cost of missing a risky borrower. Threshold tuning explicitly controls the precision–recall trade-off.

**Why have a validation set?**  
If we chose the threshold using the test set, the test result would no longer be an unbiased estimate of final performance.

**Why can’t SVM directly give the same default probability?**  
A standard Linear SVM produces a decision score, not a calibrated probability. Logistic Regression is therefore the natural final scoring model in this project.